In [ ]:
# ============================================================
# PASO 1: IMPORTAR LIBRERÍAS
# Descripción: cargamos las herramientas necesarias para
# manipular y analizar los microdatos ENDES 2023.
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


In [ ]:
# ============================================================
# PASO 2: CARGAR MÓDULOS ENDES 2023
# Descripción: cargamos los archivos RECH0 y RECH6 del año 2023.
# ============================================================

rech0 = pd.read_csv(
    "RECH0_2023.csv",
    sep=None,
    engine="python",
    encoding="latin-1"
)

rech6 = pd.read_csv(
    "RECH6_2023.csv",
    sep=None,
    engine="python",
    encoding="latin-1"
)

print("RECH0:", rech0.shape)
print("RECH6:", rech6.shape)

RECH0: (37486, 44)
RECH6: (21464, 41)


In [ ]:
# ============================================================
# PASO 3: LIMPIAR NOMBRES DE COLUMNAS
# Descripción: eliminamos caracteres extraños y espacios
# que puedan afectar la identificación de las variables.
# ============================================================

rech0.columns = (
    rech0.columns
    .str.replace('\ufeff', '', regex=False)
    .str.replace('ï»¿', '', regex=False)
    .str.strip()
)

rech6.columns = (
    rech6.columns
    .str.replace('\ufeff', '', regex=False)
    .str.replace('ï»¿', '', regex=False)
    .str.strip()
)

print("Columnas limpiadas correctamente.")

Columnas limpiadas correctamente.


In [ ]:
# ============================================================
# PASO 4: REVISAR COLUMNAS
# Descripción: comprobamos que cada variable se haya cargado
# como una columna independiente.
# ============================================================

print("Primeras columnas RECH0:")
print(rech0.columns.tolist()[:15])

print("\nPrimeras columnas RECH6:")
print(rech6.columns.tolist()[:15])

Primeras columnas RECH0:
['ID1', 'HHID', 'HV000', 'HV001', 'HV002', 'HV002A', 'HV003', 'HV004', 'HV007', 'HV008', 'HV009', 'HV010', 'HV011', 'HV012', 'HV013']

Primeras columnas RECH6:
['ID1', 'HHID', 'HC0', 'HC1', 'HC2', 'HC3', 'HC4', 'HC5', 'HC6', 'HC7', 'HC8', 'HC9', 'HC10', 'HC11', 'HC12']


In [ ]:
# ============================================================
# PASO 5: VERIFICAR VARIABLES CLAVE
# Descripción: comprobamos la existencia de las variables
# necesarias para integrar y filtrar la población.
# ============================================================

print("HHID en RECH0:", "HHID" in rech0.columns)
print("HV024 en RECH0:", "HV024" in rech0.columns)

print("HHID en RECH6:", "HHID" in rech6.columns)
print("HC1 en RECH6:", "HC1" in rech6.columns)
print("HC57 en RECH6:", "HC57" in rech6.columns)

HHID en RECH0: True
HV024 en RECH0: True
HHID en RECH6: True
HC1 en RECH6: True
HC57 en RECH6: True


In [ ]:
# ============================================================
# PASO 6: VERIFICAR EL AÑO
# Descripción: comprobamos que los archivos cargados
# correspondan al año 2023.
# ============================================================

if "ID1" in rech0.columns:
    print("Años encontrados en RECH0:")
    print(rech0["ID1"].value_counts(dropna=False))

if "ID1" in rech6.columns:
    print("\nAños encontrados en RECH6:")
    print(rech6["ID1"].value_counts(dropna=False))

Años encontrados en RECH0:
ID1
2023    37486
Name: count, dtype: int64

Años encontrados en RECH6:
ID1
2023    21464
Name: count, dtype: int64


In [ ]:
# ============================================================
# PASO 7: VERIFICAR DEPARTAMENTO
# Descripción: revisamos los códigos de departamento y
# confirmamos la presencia del código 21 para Puno.
# ============================================================

print("Códigos de departamento encontrados:")
print(rech0["HV024"].value_counts().sort_index())

print("\nRegistros con HV024 = 21:")
print((rech0["HV024"] == 21).sum())

Códigos de departamento encontrados:
HV024
1     1508
2     1320
3     1341
4     1324
5     1506
6     1382
7     1383
8     1301
9     1425
10    1510
11    1331
12    1303
13    1359
14    1373
15    4893
16    1346
17    1302
18    1305
19    1304
20    1365
21    1315
22    1347
23    1309
24    1316
25    1318
Name: count, dtype: int64

Registros con HV024 = 21:
1315


In [ ]:
# ============================================================
# PASO 8: INTEGRAR MÓDULOS
# Descripción: unimos la información del hogar y del niño
# utilizando HHID como identificador común.
# ============================================================

endes_2023 = pd.merge(
    rech0,
    rech6,
    on="HHID",
    how="inner",
    suffixes=("_hogar", "_nino")
)

print("Base integrada ENDES 2023:")
print(endes_2023.shape)

Base integrada ENDES 2023:
(21464, 84)


In [ ]:
# ============================================================
# PASO 9: FILTRAR REGIÓN PUNO
# Descripción: conservamos únicamente los registros
# pertenecientes al departamento de Puno.
# ============================================================

datos_puno_2023 = endes_2023[
    endes_2023["HV024"] == 21
].copy()

print("Registros de Puno:")
print(datos_puno_2023.shape)

Registros de Puno:
(588, 84)


In [ ]:
# ============================================================
# PASO 10: FILTRAR NIÑOS DE 6 A 35 MESES
# Descripción: seleccionamos únicamente la población objetivo
# establecida para la investigación.
# ============================================================

datos_6_35_2023 = datos_puno_2023[
    (datos_puno_2023["HC1"] >= 6) &
    (datos_puno_2023["HC1"] <= 35)
].copy()

print("Niños de 6 a 35 meses:")
print(datos_6_35_2023.shape)

print("Edad mínima:", datos_6_35_2023["HC1"].min())
print("Edad máxima:", datos_6_35_2023["HC1"].max())

Niños de 6 a 35 meses:
(289, 84)
Edad mínima: 6
Edad máxima: 35


In [ ]:
# ============================================================
# PASO 11: REVISAR VARIABLE DE ANEMIA
# Descripción: observamos los códigos de HC57 antes de
# construir la variable objetivo binaria.
# ============================================================

print("Distribución de HC57:")

print(
    datos_6_35_2023["HC57"]
    .value_counts(dropna=False)
    .sort_index()
)

Distribución de HC57:
HC57
1      4
2    103
3     88
4     82
9     12
Name: count, dtype: int64


In [ ]:
# ============================================================
# PASO 12: CREAR VARIABLE OBJETIVO
# Descripción: eliminamos los registros no válidos de HC57
# y convertimos la anemia en una variable binaria.
# 1 = presenta anemia
# 0 = no presenta anemia
# ============================================================

datos_2023 = datos_6_35_2023.dropna(
    subset=["HC57"]
).copy()

datos_2023 = datos_2023[
    datos_2023["HC57"] != 9
].copy()

datos_2023["target_anemia"] = np.where(
    datos_2023["HC57"].isin([1, 2, 3]),
    1,
    0
)

print("Distribución de target_anemia:")
print(datos_2023["target_anemia"].value_counts())

Distribución de target_anemia:
target_anemia
1    195
0     82
Name: count, dtype: int64


In [ ]:
# ============================================================
# PASO 13: RESUMEN DE ANEMIA
# Descripción: calculamos la frecuencia y porcentaje de niños
# con anemia y sin anemia en ENDES 2023.
# ============================================================

resumen_anemia_2023 = (
    datos_2023["target_anemia"]
    .value_counts()
    .rename(index={1: "Con anemia", 0: "Sin anemia"})
    .to_frame("Frecuencia")
)

resumen_anemia_2023["Porcentaje"] = (
    resumen_anemia_2023["Frecuencia"]
    / len(datos_2023)
    * 100
).round(2)

resumen_anemia_2023

,Frecuencia,Porcentaje
target_anemia,,
Con anemia,195,70.4
Sin anemia,82,29.6


In [ ]:
# ============================================================
# PASO 14: RESUMEN FINAL
# Descripción: mostramos las principales características
# de la base final ENDES 2023 para Puno.
# ============================================================

print("RESUMEN ENDES 2023 - PUNO")
print("-" * 40)

print("Total de registros:", len(datos_2023))
print("Total de variables:", datos_2023.shape[1])

print("\nEdad mínima:", datos_2023["HC1"].min(), "meses")
print("Edad máxima:", datos_2023["HC1"].max(), "meses")

print("\nDistribución de anemia:")
print(resumen_anemia_2023)

RESUMEN ENDES 2023 - PUNO
----------------------------------------
Total de registros: 277
Total de variables: 85

Edad mínima: 6 meses
Edad máxima: 35 meses

Distribución de anemia:
               Frecuencia  Porcentaje
target_anemia                        
Con anemia            195        70.4
Sin anemia             82        29.6


In [ ]:
# ============================================================
# PASO 15: EXPORTAR A EXCEL
# Descripción: guardamos el dataset final 2023 en formato
# Excel para que cada variable aparezca en su propia columna.
# ============================================================

nombre_archivo = "ENDES_PUNO_ANEMIA_2023.xlsx"

datos_2023.to_excel(
    nombre_archivo,
    index=False
)

print("Archivo generado correctamente:")
print(nombre_archivo)

Archivo generado correctamente:
ENDES_PUNO_ANEMIA_2023.xlsx
